# 12. Multi-Agent Orchestration (코드 기반 멀티에이전트 오케스트레이션)

**04_Handoffs**에서는 LLM이 자율적으로 핸드오프를 결정하는 **LLM 기반 오케스트레이션**을 배웠습니다.
이번에는 **코드로 직접** 에이전트 흐름을 제어하는 패턴을 학습합니다.

### LLM 기반 vs 코드 기반 오케스트레이션

| 구분 | LLM 기반 (Handoffs) | 코드 기반 |
|------|-------------------|----------|
| 제어 주체 | LLM이 자율 판단 | 개발자가 코드로 제어 |
| 예측 가능성 | 낮음 (LLM 판단에 의존) | 높음 (결정적 흐름) |
| 유연성 | 높음 | 보통 |
| 비용/속도 | 변동적 | 예측 가능 |
| 적합한 경우 | 복잡한 대화, 분류가 어려운 경우 | 파이프라인, 배치 처리, 정해진 워크플로우 |

### 주요 패턴
1. **순차 체이닝 (Sequential Chaining)**: A → B → C
2. **병렬 실행 (Parallelization)**: A, B, C 동시 실행
3. **분류 → 라우팅 (Classification + Routing)**: 분류 후 전문 에이전트에 전달

## 1. 순차 체이닝 (Sequential Chaining)

에이전트A의 출력을 에이전트B의 입력으로 전달하는 파이프라인입니다.
각 단계가 순서대로 실행됩니다.

```
[작성 에이전트] → 초안 → [편집 에이전트] → 수정본 → [번역 에이전트] → 최종 결과
```

In [ ]:
# Step 1: 글 작성 에이전트
# Step 2: 편집 에이전트
# Step 3: 번역 에이전트
# 순차 체이닝 실행
    # Step 1: 글 작성
    # Step 2: 편집 (이전 단계의 출력을 입력으로 전달)
    # Step 3: 번역 (이전 단계의 출력을 입력으로 전달)

## 2. 병렬 실행 (Parallelization)

서로 독립적인 에이전트를 `asyncio.gather()`로 **동시에** 실행하여 시간을 절약합니다.

```
        ┌→ [감성 분석 에이전트] → 결과1   ─┐
[입력] ──┼→ [키워드 추출 에이전트] → 결과2 ─┼→ [종합]
        └→ [요약 에이전트] → 결과3   ─────┘
```

In [ ]:
# 3개의 독립적인 분석 에이전트
# asyncio.gather로 3개 에이전트를 동시에 실행

## 3. 분류 → 라우팅 (Classification + Routing)

**Structured Output**으로 입력을 분류한 후, 결과에 따라 적절한 전문 에이전트를 실행합니다.

```
[입력] → [분류 에이전트] → 카테고리 → [전문 에이전트 선택] → [전문 에이전트 실행] → [응답]
```

In [ ]:
# 분류 결과를 위한 Pydantic 모델
class RequestClassification(BaseModel):
# 분류 에이전트
# 전문 에이전트들
# 카테고리 → 에이전트 매핑

In [ ]:
# 분류 → 라우팅 실행
        # Step 1: 분류
        # Step 2: 라우팅 - 분류 결과에 따라 적절한 에이전트 선택
        # Step 3: 전문 에이전트 실행

In [ ]:
# 테스트 1: 기술지원 질문

In [ ]:
# 테스트 2: 결제 질문

In [ ]:
# 테스트 3: 일반 질문

## 4. 종합 패턴: 병렬 + 순차 + 체이닝 조합

실전에서는 여러 패턴을 **조합**하여 사용합니다.

```
[고객 리뷰] ──→ [병렬 분석] ──→ [종합 에이전트] ──→ [최종 보고서]
                 ├ 감성 분석
                 ├ 키워드 추출
                 └ 요약
```

In [ ]:
# 종합 보고서를 작성하는 에이전트
    # Step 1: 병렬 분석 (동시 실행)
    # Step 2: 순차 체이닝 - 분석 결과를 종합

---------------------
### 정리

| 패턴 | 코드 | 적합한 경우 |
|------|------|-----------|
| 순차 체이닝 | `result1 = run(A, input)` → `result2 = run(B, result1.final_output)` | 파이프라인, 단계별 처리 |
| 병렬 실행 | `await asyncio.gather(run(A, x), run(B, x), run(C, x))` | 독립적 분석, 시간 절약 |
| 분류 → 라우팅 | `Structured Output`으로 분류 → `agent_map[category]` | 고객 문의 분류, 의도 분석 |
| 조합 패턴 | 병렬 + 순차를 결합 | 실전 워크플로우 |

**핵심 포인트:**
- `trace()`로 전체 워크플로우를 하나의 트레이스로 묶어 추적
- `asyncio.gather()`로 독립적인 작업을 병렬 실행하여 성능 향상
- `Structured Output (output_type)`으로 결정적(deterministic) 라우팅 구현
- 코드 기반 오케스트레이션은 **예측 가능성**과 **비용 제어**에 유리